# Oasis complaint classifier: BERT fine-tuning (Arabic + Franco + English)

**What this notebook does:** trains a BERT model that reads a complaint and returns its category. **X = the complaint text, y = `category_id`** (16 categories). At the end it downloads **`oasis_bert_model.zip`**: unzip it inside the platform's `ml/artifacts/` folder and the website uses it automatically.

**How to run:**
1. `Runtime > Change runtime type > T4 GPU`.
2. `Runtime > Run all`.
3. Put the **data** zip in Colab: drag it into the Files panel (folder icon on the left) and wait for the upload to finish, or use the button in step 2. Any of these works, whatever you renamed it to:
   - `Oasis_READY_CLEAN_DATA_v4.zip` (77 MB)
   - `Oasis_Service_Intelligence_FINAL.zip` (116 MB)
   - **Not** `Oasis_Complaints_Platform...zip`: that is the website. It has no data, only the website code (the notebook still falls back to the website's copy of the training sentences if you upload it by mistake, and says so).
4. Wait for the download of `oasis_bert_model.zip` (about 5 to 10 minutes on a T4).
5. Scroll to **step 12, RESULTS DISPLAY**: accuracy cards, learning curve, F1 per category, confusion matrix, and the sentences it got wrong. The charts are also saved in `/content/results`.

**What the notebook reads from the zip** (it finds the files itself, the zip's name does not matter):

| Data zip | Text (X) | Category (y) |
|---|---|---|
| `Oasis_READY_CLEAN_DATA_v4.zip` | `01_CLEAN_SOURCE_CSV/service_requests.csv`, column `description_ar` | `02_SUPPORTING_CSV/complaint_training_labels.csv`, column `category_id`, joined on `request_id` |
| `Oasis_Service_Intelligence_FINAL.zip` | `data/prepared/service_requests_prepared.csv`, column `description_ar` | same file, column `category_id` |

Both have the same 44,817 complaints. They repeat about 800 sentence templates, so after removing exact repeats about 1,500 different sentences remain. The data **does contain Franco and English** (4,965 complaints written only in Latin letters, 5,675 mixed), but only 7 different Franco sentences. So the 750 extra hand-written sentences (Egyptian Arabic, Franco, English) from the platform's `ml/augment_*.csv` are embedded below and added to training.

**Franco:** Arabic BERT models were trained on Arabic script, so each text gets the Arabic words of its Franco/English domain words added (`"el zebala" -> "el zebala الزباله"`), the same rule the website uses (`ml/franco.py`). The model sees both.

All data is SYNTHETIC (Oasis District 2025 training kit), not real ACUD data.

## 1. Settings and libraries

In [ ]:
# Arabic BERT trained on dialectal tweets (strong on Egyptian Arabic). Alternatives that also work:
#   "aubmindlab/bert-base-arabertv02-twitter", "CAMeL-Lab/bert-base-arabic-camelbert-da", "xlm-roberta-base"
MODEL_NAME = "UBC-NLP/MARBERTv2"
MAX_LEN = 96          # complaints are short; 96 tokens covers all of them
EPOCHS = 8
BATCH_SIZE = 32
LR = 3e-5
VAL_SHARE = 0.15      # share of sentence templates kept aside to measure the model honestly
SEED = 42

!pip -q install -U transformers scikit-learn pandas
import os, io, re, json, gzip, base64, random, zipfile, shutil, time
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
from IPython.display import display

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def find_data_files():
    """Data zips/CSVs already in Colab's Files panel (/content) or in a mounted Google Drive, best guess first."""
    hits = []
    for root, max_depth in (("/content", 2), ("/content/drive/MyDrive", 3)):
        if not os.path.isdir(root):
            continue
        for dirpath, dirnames, filenames in os.walk(root):
            depth = dirpath[len(root):].count(os.sep)
            dirnames[:] = [] if depth >= max_depth else [d for d in dirnames if d not in ("sample_data", "drive", "bert", ".config")]
            for f in filenames:
                low = f.lower()
                if (low.endswith(".zip") or low.endswith(".csv")) and not low.startswith("oasis_bert_model"):
                    hits.append(os.path.join(dirpath, f))
    def rank(path):
        low = os.path.basename(path).lower()
        is_data = any(k in low for k in ("service_intelligence", "ready_clean_data", "service_requests"))
        is_website = "platform" in low or "complaints_platform" in low
        return (not is_data, is_website, -os.path.getsize(path))
    return sorted(hits, key=rank)
print("device:", DEVICE, "| GPU:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "none (Runtime > Change runtime type > T4 GPU)")

## 2. Put the data zip in Colab
**Easiest and most reliable (recommended for a 77 to 116 MB file):** click the **folder icon** on the left of Colab, drag `Oasis_Service_Intelligence_FINAL.zip` (or `Oasis_READY_CLEAN_DATA_v4.zip`) into that panel, and **wait until the upload circle at the bottom of the panel finishes**. Then run this cell. It finds the file by itself, whatever its name.

If no file is there, this cell opens a *Choose Files* button instead. Wait until it shows 100% before running the next cell.

In [ ]:
from google.colab import files
candidates = find_data_files()
if candidates:
    DATA_FILE = candidates[0]
    print("using:", DATA_FILE)
else:
    print("No data file in the Files panel yet. Choose it now:")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("Nothing was uploaded. Drag the data zip into the Files panel (folder icon on the left), wait for it to finish, and run this cell again.")
    DATA_FILE = next(iter(uploaded))
size_mb = os.path.getsize(DATA_FILE) / 1e6
print(f"data file: {DATA_FILE} ({size_mb:.1f} MB)")
if size_mb < 10:
    print("WARNING: this file is small. The data zips are 77 MB (READY_CLEAN_DATA_v4) or 116 MB (FINAL). "
          "If it is the website zip or an unfinished upload, replace it and run this cell again.")

## 3. Read the text (X) and the category (y)
The notebook looks inside the upload for one of the known layouts and tells you which one it used.

In [ ]:
# Arabic category names (and their known spellings) -> category_id, from the kit's complaint_categories.csv
# and complaint_category_aliases.csv. Used only when a file has the Arabic name but no category_id.
NAME_TO_ID = {}

def _read(z, name, **kw):
    return pd.read_csv(io.BytesIO(z.read(name)), encoding="utf-8-sig", **kw)

def _find(names, suffix):
    hits = [n for n in names if n.lower().endswith(suffix.lower()) and "__macosx" not in n.lower()]
    return hits[0] if hits else None

def _from_table(t, where):
    if "category_id" not in t.columns and "category" in t.columns:
        t = t.assign(category_id=t["category"].astype(str).str.strip().map(NAME_TO_ID))
    missing = {"description_ar", "category_id"} - set(t.columns)
    if missing:
        raise ValueError(f"{where} has no column(s) {sorted(missing)}; its columns are {list(t.columns)}")
    return t[["description_ar", "category_id"]].dropna()

def load_from_zip(z, label):
    """Returns (training table, scenario test table or None, description of what was used)."""
    names = z.namelist()
    # Layout 1: Oasis_Service_Intelligence_FINAL
    f = _find(names, "data/prepared/service_requests_prepared.csv")
    if f:
        kit = _read(z, f, usecols=["description_ar", "category_id"])
        s = _find(names, "scenario_complaints_with_answer_keys_SYNTHETIC.csv")
        sc = _read(z, s).rename(columns={"answer_key_category_id": "category_id"}) if s else None
        return kit, sc, f"FINAL package: {f}"
    # Layout 2: Oasis_READY_CLEAN_DATA_v4 (text and label are in two files, joined on request_id)
    f, lab = _find(names, "01_CLEAN_SOURCE_CSV/service_requests.csv"), _find(names, "complaint_training_labels.csv")
    if f and lab:
        kit = _read(z, f, usecols=["request_id", "description_ar"]).merge(_read(z, lab), on="request_id", how="inner")
        sc = None
        s, sl = _find(names, "02_SUPPORTING_CSV/scenario_complaints.csv"), _find(names, "test_complaint_category_labels.csv")
        if s and sl:
            sc = _read(z, s, usecols=["request_id", "description_ar"]).merge(_read(z, sl), on="request_id", how="inner")
        return kit, sc, f"READY_CLEAN_DATA_v4: {f} + {lab}"
    # A zip inside the zip
    for inner in [n for n in names if n.lower().endswith(".zip")]:
        with zipfile.ZipFile(io.BytesIO(z.read(inner))) as z2:
            try:
                return load_from_zip(z2, f"{label} > {inner}")
            except FileNotFoundError:
                pass
    # Layout 3: the website zip (Oasis_Complaints_Platform...): its copy of the kit's unique sentences
    f = _find(names, "ml/kit_training_texts.csv")
    if f:
        print("NOTE: this is the WEBSITE zip, not the data zip. Using its copy of the kit's training sentences "
              "(same sentences, already de-duplicated). Upload the data zip next time to also get the scenario test.")
        return _read(z, f), None, f"website zip: {f}"
    csvs = [n for n in names if n.lower().endswith(".csv")][:15]
    raise FileNotFoundError(
        f"{label} does not contain the complaint data.\n"
        "Expected one of:\n  data/prepared/service_requests_prepared.csv  (Oasis_Service_Intelligence_FINAL.zip)\n"
        "  01_CLEAN_SOURCE_CSV/service_requests.csv + 02_SUPPORTING_CSV/complaint_training_labels.csv  (Oasis_READY_CLEAN_DATA_v4.zip)\n"
        f"CSV files found in it: {csvs or 'none'}")

def load_data(path):
    if zipfile.is_zipfile(path):
        with zipfile.ZipFile(path) as z:
            kit, sc, used = load_from_zip(z, path)
    else:
        kit, sc, used = pd.read_csv(path, encoding="utf-8-sig"), None, f"CSV file: {path}"
    return _from_table(kit, used), (_from_table(sc, "scenario test") if sc is not None else None), used

if "DATA_FILE" not in globals():   # step 2 was skipped or did not finish
    _found = find_data_files()
    if not _found:
        raise RuntimeError("No data file found. Run step 2 first: drag the data zip into the Files panel "
                           "(folder icon on the left), wait for the upload to finish, then run step 2 and this cell.")
    DATA_FILE = _found[0]
    print("step 2 did not run; found the data file by itself:", DATA_FILE)
kit, scenario, USED = load_data(DATA_FILE)
print("read from:", USED)
unknown = sorted(set(kit.category_id) - set(NAME_TO_ID.values()))
assert not unknown, f"unknown categories in the data: {unknown}"

def unpack(b64):
    return pd.read_csv(io.BytesIO(gzip.decompress(base64.b64decode(b64))))

EXTRA_B64 = "..."
extra = unpack(EXTRA_B64)

t = kit.description_ar.astype(str).str.replace(r"\[[A-Z_]+\]", "", regex=True)
latin, arabic = t.str.contains("[A-Za-z]"), t.str.contains("[\u0600-\u06FF]")
print(f"complaints in the data: {len(kit):,} | written only in Latin letters (Franco/English): {(latin & ~arabic).sum():,} "
      f"| mixed: {(latin & arabic).sum():,}")

# Rule agreed with the team: keep the Franco training sentences only if the data itself has Franco.
FRANCO_IN_DATA = int((latin & ~arabic).sum()) > 0
ex = extra.description_ar.astype(str)
extra_latin_only = ex.str.contains("[A-Za-z]") & ~ex.str.contains("[\u0600-\u06FF]")
if FRANCO_IN_DATA:
    print(f"The data contains Franco/English complaints -> the {int(extra_latin_only.sum())} Franco/English training sentences are KEPT.")
else:
    extra = extra[~extra_latin_only]
    print("The data has no Franco/English complaints -> Franco/English training sentences REMOVED.")
print(f"extra hand-written sentences added: {len(extra):,}")

df = pd.concat([kit, extra], ignore_index=True).dropna()
df["description_ar"] = df.description_ar.astype(str).str.strip()
df = df.drop_duplicates(["description_ar", "category_id"]).reset_index(drop=True)
print(f"unique (text, category) pairs used for training: {len(df):,}")
if scenario is not None:
    print(f"scenario test complaints (never trained on): {len(scenario):,}")
df.category_id.value_counts()

### Data overview

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
kit.category_id.value_counts().sort_values().plot.barh(ax=ax[0], color="#0f766e")
ax[0].set_title("Complaints per category in the data (y)"); ax[0].set_xlabel("complaints")
df.category_id.value_counts().sort_values().plot.barh(ax=ax[1], color="#64748b")
ax[1].set_title("Unique training sentences per category (after removing repeats)"); ax[1].set_xlabel("sentences")
plt.tight_layout(); plt.show()

## 4. Franco/English gloss (same code as the website's `ml/franco.py`)

In [ ]:
%%writefile oasis_franco.py
"""Franco-Arabic (Arabizi) and English support for the complaint classifier.

The kit's training texts are almost all Arabic script. A citizen who writes
"el zebala mtkawema" or "the sewage is overflowing" shares almost no words with
them, and Franco spellings vary a lot ("zbala", "zebala", "zibala").

Two steps close that gap, both rule-based and explainable:

1. `skeleton(word)`: a spelling-insensitive key for a Latin word. Digits become
   their usual letters, vowels are dropped and repeats collapse, so "zebala",
   "zbala" and "zibalaa" all become "zbl".
2. `gloss(text)`: every Latin word whose skeleton is in the lexicon below adds
   the matching Arabic word. The Arabic word is what the kit uses, so a Franco
   complaint reaches the same word and character features as an Arabic one.

The lexicon is hand-written domain vocabulary (one Arabic word per concept, many
Franco/English spellings). It was not mined from any test or probe set.
"""

import re

# Arabic word -> Franco / English spellings. Keep the Arabic side in the kit's own
# wording (see the most frequent words per category in the training data).
LEXICON = {
    # water
    "ميه": ["maya", "mayya", "mayy", "mya", "maia", "mayah", "maaya", "water", "mayet"],
    "مياه": ["meyah", "miyah", "meyyah", "mayah"],
    "ماسوره": ["masoura", "masora", "masoora", "masura", "mwasir", "mawaseer", "mwaser", "pipe", "pipes"],
    "تسريب": ["tasreeb", "tasrib", "tasreb", "leak", "leaking", "leaks", "leakage", "bt5or", "bete5or", "5ar"],
    "مغرقه": ["mghr2a", "mghar2a", "mgharaqa", "ghar2a", "ghar2an", "gharqa", "flooded", "flooding", "flood"],
    "مكسوره": ["maksoura", "maksora", "mkasara", "mkasar", "maksoor", "maksour", "broken", "burst"],
    "مقطوعه": ["ma2to3a", "maqtoo3a", "ma2too3a", "2ate3", "2at3a", "ati3a", "2at3", "et2at3et", "it2ata3it", "cut", "cutoff"],
    "مفيش": ["mafihash", "mafihoosh", "mafeesh", "mafeesh", "mafish", "mfeesh", "mafesh", "mafeech", "no"],
    "ضغط": ["daght", "dag5t", "da8t", "da3t", "pressure"],
    "ضعيف": ["da3eef", "da3if", "da3ef", "da3eefa", "da3ifa", "weak", "low"],
    "الحنفيه": ["7anafeya", "hanafeya", "7anafiya", "7anafya", "tap", "faucet"],
    "بتنزل": ["nazla", "naazla", "btenzel", "bitinzil", "t2ta2", "nu2ta"],
    "واطي": ["wate", "waty", "waye2", "wa6y"],
    # sewer
    "بلاعه": ["bala3a", "balla3a", "bala3", "manhole", "drain"],
    "طافحه": ["taf7a", "tf7et", "tef7et", "tafha", "tafa7et", "overflowing", "overflow"],
    "المجاري": ["magary", "magari", "majari", "sewage", "sewer"],
    "صرف": ["sarf", "sar5", "saarf"],
    "ريحه": ["ri7a", "re7a", "ri7et", "re7et", "smell", "stink", "smells"],
    # power
    "الكهربا": ["kahraba", "kahrba", "ka7raba", "kahrabaa", "electricity", "power"],
    "النور": ["nour", "nur", "noor"],
    "انقطاع": ["en2eta3", "in2ita3", "outage", "blackout"],
    # hvac
    "التكييف": ["tabreed", "tabrid", "cooling", "takyeef", "takeef", "takyif", "takief", "ac", "conditioner", "conditioning", "hvac"],
    "سخن": ["so5n", "su5n", "sokhn", "sukhn", "hot"],
    "الحر": ["7ar", "har", "heat"],
    # waste
    "الزباله": ["zebala", "zbala", "zibala", "zebbala", "zbalah", "garbage", "trash", "rubbish", "waste"],
    "الصندوق": ["sandoo2", "sondoo2", "sanadee2", "sanade2", "sandou2", "bin", "bins", "container"],
    "مليان": ["malyan", "malyana", "malian", "full"],
    # cleanliness
    "كنس": ["kans", "kanas", "kannas", "sweep", "sweeping", "sweeps"],
    "تراب": ["trab", "tarab", "torab", "dust", "dusty"],
    "مخلفات": ["mo5alafat", "m5lfat", "debris", "rubble"],
    "وسخ": ["wesi5", "wesekh", "wese5", "wasa5", "wsa5a", "wasa5a", "dirty", "filthy"],
    # roads
    "حفره": ["7ofar", "hofar", "7ofra", "hofra", "7ofa", "pothole", "potholes", "hole"],
    "مطب": ["matab", "mataab", "matabb", "bump", "speedbump"],
    "الرصيف": ["rasif", "raseef", "rasef", "sidewalk", "pavement"],
    "الاسفلت": ["asfalt", "asphalt"],
    "الطريق": ["tari2", "tari2", "taree2", "tare2", "road"],
    # parking
    "راكنه": ["rakna", "raknin", "rakneen", "raken", "parked", "parking"],
    "الركن": ["arkan", "rakn", "arken"],
    "الجراج": ["garage", "garag", "jaraj"],
    # traffic signals
    "الاشاره": ["ishara", "esharet", "esh3ara", "eshara", "ishaara", "esharat", "light", "signal", "signals"],
    "حمرا": ["7amra", "hamra", "red"],
    "الزحمه": ["zahma", "za7ma", "za7meh", "traffic", "congestion"],
    "التقاطع": ["ta2ato3", "taqato3", "junction", "intersection", "roundabout", "dawaran", "dawwaran"],
    # streetlights
    "عمود": ["3amood", "3amoud", "3amod", "3awamid", "pole", "lamppost"],
    "لمبه": ["lamba", "lamda", "lambet", "lamdat", "lamp", "bulb"],
    "مطفي": ["matfy", "mtafy", "tafy", "mtafya", "tafya", "off"],
    "ضلمه": ["dalma", "dlma", "dalmah", "dark", "darkness"],
    "الاناره": ["enara", "inara", "lighting", "streetlight", "streetlights"],
    # safety
    "غاز": ["gaz", "ghaz", "gas"],
    "دخان": ["do5an", "dokhan", "smoke"],
    "حريق": ["7aree2a", "7ari2a", "7aree2", "7ari2", "fire", "burning"],
    "حرامي": ["7aramy", "7arami", "haramy", "thief", "thieves"],
    "سرق": ["yesra2", "bysra2", "sor2o", "bysor2o", "break", "sara2", "sere2", "sara2o", "itsara2", "stole", "stolen", "theft", "robbed"],
    "مريب": ["mashbooh", "mashbooha", "mashboha", "mureeb", "mreeb", "ghareeb", "ghareeba", "ghorab", "suspicious", "stranger", "strangers"],
    "سكينه": ["sekeena", "sekina", "sakeena", "knife"],
    "بيهدد": ["byhadded", "bihaded", "beyhadded", "threat", "threatening", "threatens"],
    "ضاله": ["dalla", "dala", "kelab", "klab", "stray", "dogs"],
    "سلك": ["selk", "silk", "wire", "wires", "cable", "makshoof", "exposed"],
    "خطر": ["5atar", "khatar", "5atr", "danger", "dangerous"],
    # noise
    "دوشه": ["fara7", "far7", "dj", "dawsha", "dosha", "dawshah", "noise", "noisy", "loud"],
    "صوت": ["kalaksat", "klaks", "horns", "sot", "sout", "sawt", "sound"],
    "ورشه": ["warsha", "wars", "wersha", "workshop"],
    "مكبرات": ["sammaat", "sama3at", "samaat", "speakers", "music"],
    "عالي": ["3aly", "3ali", "3alya", "3alia"],
    "الكافيه": ["ka7ok", "kahwa", "2ahwa", "cafe", "coffee"],
    # parks
    "الرشاشات": ["rashashat", "sprinkler", "sprinklers", "rai", "ray", "by2assi", "watering", "waters"],
    "الحديقه": ["7adi2a", "hadi2a", "7adee2a", "park", "garden"],
    "الجنينه": ["geneena", "gnena", "gneina", "genena", "gene3na"],
    "الزرع": ["zar3", "zr3", "plants", "grass"],
    "ناشف": ["nashef", "nashf", "dry", "dried"],
    "العاب": ["al3ab", "ar2al", "playground", "swings", "swing"],
    "شجره": ["shagara", "shagar", "shgara", "tree", "trees"],
    "النجيل": ["nageel", "nagil", "lawn"],
    # app
    "صاله": ["sala", "sa2a", "2a3a", "hall", "lobby"],
    "التطبيق": ["update", "error", "tatbee2", "tatbe2", "tatbi2", "app", "application", "abblication", "ablikeshn"],
    "بيقفل": ["by2fel", "bey2fel", "bi2fil", "crash", "crashes", "crashing"],
    "دخول": ["login", "dokhool", "do5ool", "signin"],
    "رساله": ["resala", "risala", "otp", "sms", "message", "code"],
}

_DIGITS = {"2": "", "3": "3", "5": "kh", "6": "t", "7": "h", "8": "gh", "9": "k"}
_LATIN_WORD = re.compile(r"[a-z0-9]+")
_ONLY_DIGITS = re.compile(r"^[0-9]+$")


def skeleton(word: str) -> str:
    w = word.lower()
    if _ONLY_DIGITS.match(w):
        return ""
    w = "".join(_DIGITS.get(ch, ch) for ch in w)
    w = w.replace("q", "k").replace("j", "g").replace("ch", "sh").replace("ou", "u")
    w = re.sub(r"[aeiouy]", "", w)
    return re.sub(r"(.)\1+", r"\1", w)


def _build_index():
    """skeleton -> Arabic word, leaving out skeletons that point at two concepts."""
    targets = {}
    for arabic, spellings in LEXICON.items():
        for s in spellings:
            targets.setdefault(skeleton(s), set()).add(arabic)
    return {k: v.pop() for k, v in targets.items() if len(k) >= 3 and len(v) == 1}


_INDEX = _build_index()
_EXACT = {s: ar for ar, spellings in LEXICON.items() for s in spellings}


def _lookup(word: str):
    if word in _EXACT:
        return _EXACT[word]
    for prefix in ("el", "al", "fel", "bel", "wel", "w", "b", "f", "l"):
        if word.startswith(prefix) and len(word) - len(prefix) >= 3 and word[len(prefix):] in _EXACT:
            return _EXACT[word[len(prefix):]]
    return _INDEX.get(skeleton(word))


def gloss(text: str) -> str:
    """Arabic words for the Franco/English domain words found in `text` ('' if none)."""
    words = _LATIN_WORD.findall(str(text).lower())
    hits = [ar for ar in (_lookup(w) for w in words) if ar]
    return " ".join(dict.fromkeys(hits))


def latin_skeleton_text(text: str) -> str:
    """The text's Latin words as skeletons, for a spelling-insensitive feature."""
    keys = [skeleton(w) for w in _LATIN_WORD.findall(str(text).lower())]
    return " ".join(k for k in keys if k) or "nolatin"

In [ ]:
import importlib, oasis_franco; importlib.reload(oasis_franco)
from oasis_franco import gloss

def model_input(text):
    g = gloss(text)
    return f"{text} {g}".strip()

for t in ["el zebala mtkawema gamb el bawaba", "fe ri7et gaz fel 3emara", "the AC is broken", "المية مقطوعة"]:
    print(t, "->", model_input(t))

## 5. Labels and an honest split
The kit repeats each sentence with a phone number or street swapped in. Splitting by the first three words of the sentence keeps one template out of both sides, so the validation score is not inflated.

In [ ]:
LABELS = sorted(df.category_id.unique())
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}

def normalize_ar(text):
    text = re.sub(r"[\u064B-\u0652\u0640]", "", str(text))
    text = re.sub("[إأآٱ]", "ا", text).replace("ى", "ي").replace("ة", "ه")
    text = re.sub(r"[^\w\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip().lower()

groups = df.description_ar.map(lambda t: " ".join(normalize_ar(t.split("،")[0]).split()[:3]))
tr_idx, va_idx = next(GroupShuffleSplit(n_splits=1, test_size=VAL_SHARE, random_state=SEED).split(df, groups=groups))
train_df, val_df = df.iloc[tr_idx], df.iloc[va_idx]
print(f"{len(LABELS)} labels | train {len(train_df):,} | validation {len(val_df):,} (templates never shared)")

## 6. Model and training loop

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ComplaintSet(Dataset):
    def __init__(self, texts, labels=None):
        self.enc = tokenizer([model_input(t) for t in texts], truncation=True, max_length=MAX_LEN, padding="max_length")
        self.labels = labels
    def __len__(self):
        return len(self.enc["input_ids"])
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[i])
        return item

def new_model():
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(LABELS), id2label=id2label, label2id=label2id,
        ignore_mismatched_sizes=True,  # a fresh 16-way head even if the checkpoint already has one
    ).to(DEVICE)

def class_weights(labels):
    counts = np.bincount(labels, minlength=len(LABELS)).astype(float)
    w = counts.sum() / (len(LABELS) * np.maximum(counts, 1))
    return torch.tensor(w, dtype=torch.float, device=DEVICE)

@torch.no_grad()
def predict_proba(model, texts, batch=128):
    model.eval(); out = []
    for s in range(0, len(texts), batch):
        enc = tokenizer([model_input(t) for t in texts[s:s + batch]], truncation=True, max_length=MAX_LEN,
                        padding=True, return_tensors="pt").to(DEVICE)
        out.append(torch.softmax(model(**enc).logits, dim=-1).cpu().numpy())
    return np.vstack(out)

def train(texts, labels, epochs, val=None):
    model = new_model()
    loader = DataLoader(ComplaintSet(texts, labels), batch_size=BATCH_SIZE, shuffle=True)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps = len(loader) * epochs
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights(np.array(labels)))
    history = []
    for ep in range(1, epochs + 1):
        model.train(); total = 0.0; t0 = time.time()
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            y = batch.pop("labels")
            loss = loss_fn(model(**batch).logits, y)
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); opt.zero_grad(); total += loss.item()
        row = {"epoch": ep, "loss": round(total / len(loader), 4), "sec": round(time.time() - t0)}
        if val is not None:
            pred = predict_proba(model, val[0]).argmax(1)
            row["val_acc"] = round(accuracy_score(val[1], pred), 4)
            row["val_macro_f1"] = round(f1_score(val[1], pred, average="macro"), 4)
        history.append(row); print(row)
    return model, history

## 7. Round 1: train on 85%, measure on the unseen 15%
This picks the best number of epochs and gives the honest score.

In [ ]:
y_tr = train_df.category_id.map(label2id).tolist()
y_va = val_df.category_id.map(label2id).tolist()
model, history = train(train_df.description_ar.tolist(), y_tr, EPOCHS, val=(val_df.description_ar.tolist(), y_va))
best = max(history, key=lambda r: r["val_macro_f1"])
BEST_EPOCHS = best["epoch"]
print(f"best epoch: {BEST_EPOCHS} | validation macro-F1 {best['val_macro_f1']} | accuracy {best['val_acc']}")
val_pred = predict_proba(model, val_df.description_ar.tolist()).argmax(1)
report = pd.DataFrame(classification_report(y_va, val_pred, labels=list(range(len(LABELS))), target_names=LABELS,
                                            zero_division=0, output_dict=True)).T.round(3)
validation = {"macro_f1": best["val_macro_f1"], "accuracy": best["val_acc"], "best_epochs": BEST_EPOCHS, "history": history}
del model; torch.cuda.empty_cache()

### RESULTS 1: accuracy of the model on sentences it never saw

In [ ]:
# ===== RESULTS 1: how accurate is the model? (sentences it never saw during training) =====
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, HTML
import os; os.makedirs("results", exist_ok=True)
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# Predictions of the round-1 model on the 15% it never trained on (honest score).
if "val_pred" in globals():
    _vp = np.asarray(val_pred)
elif "pred" in globals() and len(pred) == len(y_va):
    _vp = np.asarray(pred)
else:
    raise RuntimeError("Run this cell right after step 7 (Round 1), before step 8.")
_acc, _f1 = accuracy_score(y_va, _vp), f1_score(y_va, _vp, average="macro")

display(HTML(f"""
<div style="font-family:sans-serif;display:flex;gap:16px;flex-wrap:wrap;margin:8px 0">
  <div style="background:#0f766e;color:white;border-radius:12px;padding:16px 24px;min-width:180px">
    <div style="font-size:14px;opacity:.85">Accuracy</div><div style="font-size:40px;font-weight:700">{_acc:.1%}</div></div>
  <div style="background:#115e59;color:white;border-radius:12px;padding:16px 24px;min-width:180px">
    <div style="font-size:14px;opacity:.85">Macro-F1</div><div style="font-size:40px;font-weight:700">{_f1:.3f}</div></div>
  <div style="background:#334155;color:white;border-radius:12px;padding:16px 24px;min-width:180px">
    <div style="font-size:14px;opacity:.85">Test sentences</div><div style="font-size:40px;font-weight:700">{len(y_va)}</div></div>
  <div style="background:#334155;color:white;border-radius:12px;padding:16px 24px;min-width:180px">
    <div style="font-size:14px;opacity:.85">Best epoch</div><div style="font-size:40px;font-weight:700">{BEST_EPOCHS}</div></div>
</div>
<p style="font-family:sans-serif;color:#475569">Measured on {len(y_va)} sentences the model never saw (no sentence template shared with training).</p>
"""))

# 1) Learning curve
_h = pd.DataFrame(history)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(_h.epoch, _h.loss, marker="o", color="#64748b"); ax[0].set_title("Training loss (lower is better)"); ax[0].set_xlabel("epoch")
ax[1].plot(_h.epoch, _h.val_acc, marker="s", color="#d97706", label="accuracy")
ax[1].plot(_h.epoch, _h.val_macro_f1, marker="o", color="#0f766e", label="macro-F1")
ax[1].axvline(BEST_EPOCHS, ls="--", color="grey"); ax[1].set_ylim(0, 1.02); ax[1].legend()
ax[1].set_title("Score on unseen sentences after each epoch"); ax[1].set_xlabel("epoch")
plt.tight_layout(); fig.savefig("results/learning_curve.png", dpi=150); plt.show()

# 2) Accuracy per category
_rep = pd.DataFrame(classification_report(y_va, _vp, labels=list(range(len(LABELS))), target_names=LABELS,
                                          zero_division=0, output_dict=True)).T
_pc = _rep.loc[LABELS].sort_values("f1-score")
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(_pc.index, _pc["f1-score"], color=["#dc2626" if v < 0.7 else "#0f766e" for v in _pc["f1-score"]])
for i, v in enumerate(_pc["f1-score"]):
    ax.text(v + 0.01, i, f"{v:.2f}", va="center")
ax.set_xlim(0, 1.1); ax.set_title("F1 per category (red = below 0.70)"); plt.tight_layout(); fig.savefig("results/f1_per_category.png", dpi=150); plt.show()

# 3) Confusion matrix
_cm = confusion_matrix(y_va, _vp, labels=list(range(len(LABELS))))
fig, ax = plt.subplots(figsize=(10, 9))
ax.imshow(_cm, cmap="Greens")
ax.set_xticks(range(len(LABELS))); ax.set_xticklabels(LABELS, rotation=60, ha="right")
ax.set_yticks(range(len(LABELS))); ax.set_yticklabels(LABELS)
for i in range(len(LABELS)):
    for j in range(len(LABELS)):
        if _cm[i, j]:
            ax.text(j, i, _cm[i, j], ha="center", va="center", color="white" if _cm[i, j] > _cm.max() / 2 else "black")
ax.set_xlabel("predicted"); ax.set_ylabel("true category"); ax.set_title("Confusion matrix (diagonal = correct)")
plt.tight_layout(); fig.savefig("results/confusion_matrix.png", dpi=150); plt.show()

display(_rep.round(3).rename(columns={"support": "sentences"}))
RESULTS = {"accuracy": round(_acc, 4), "macro_f1": round(_f1, 4), "test_sentences": len(y_va), "best_epoch": int(BEST_EPOCHS)}

## 8. Round 2: the final model, trained on ALL the data

In [ ]:
final_model, _ = train(df.description_ar.tolist(), df.category_id.map(label2id).tolist(), BEST_EPOCHS)

## 9. Tests the model never saw
- 64 new Franco/English sentences written after the TF-IDF model was finished (the TF-IDF 1.2.0 model scores 85% Franco / 94% English on them).
- The kit's scenario complaints (only if the zip was uploaded).

In [ ]:
PROBE_B64 = "H4sIAAAAAAAC/31XwZKrOAy8v6/w7V1ygpql9mumBBbgwtis7QxLvn5bdmAgL7OXSQbcttVqtZTE/6ZbR4kHH7ZPo2+xC2ZJv9gqClp5TbPC95ZaNYwUKnKkZtpIOdbMPf5xKlKdmN1txTbh0zJNtz6Q6/yvntUHpQJIFLhSPTYb+GECybarj4ne4PAqY1q8iKOcsVK0lNENv4WkkQP/jorUctfaskojTnb8xUHpYDiqlju6R5YVZmFlohK8ccN5N3aDNXH8NeNYS9WGuPZ4I+JvFa34Ws/Cz2SfSH9PNPDr5VeqwE9NTmcGre8mNXnrJZxaSKP0E74BzT1vQnWbHD3Y4joNDQQ6Lce3sEQLog+sOj8vlhPbDYFvyjjl70HFFJhfDtyDPdHNw4YzTFIBF+Aaid9PWwLHeA/8af16umqkaSRwBHpUu+FvTRZBR3m476upFq38z0aahjEd61uqQB83XHLfUp0pTNAANwdr7/bZn0l2k/cKr/CpemMBHyEKctM7+ImJlixSAy4q2iSLcrKkHmTk7AXT0C3yymE/NN85Uuixva1zSTzLZ6C5LVHFgfUVhXJKVF660UOwM03R+3BdJZfOD5Q1LofVUieiVfcFqUVseUVJbkGeYploDNSibnL11CUlEgwSm6h2dFv8+k6APQolih76KGWaBcxQRWZAk53fI2cRDZK2nytMzC24a/LBwuvfb4ErI1VRFMBdCqYzKQtXYlszOSJqf3f6ij6FmmhCsfaF0DDRY5MCxIM+P2kR6/hF3SlGIATQ8lZBb3WRnqgXiUjUvi5vKIDHhxTF4Ff6PqjQs5+fCwEUI732soXE0nlvJXdIo+/7PcLI4ct0rIB4Qk5xPVgEKaloK/EDu4kFQ9tWTvWx9TUEHRN/elhdf6qFhlbeOFfVsclfPTU5xkDR9D8ABZDXzynQLDaEreA7RT3wM++rH6CtccWE9hcSrThghGOCJA7BtJZf0ad4n6WGzrFSoArkgliK8gDiK+7fwaqd1EOMpzvnkNQsBOH7bM/WAUMSA3oDbEjL6i2bzoaW0bKj0qfKVd6BnG+93pCwqOLKS8ppDB5bIaUr8xQvqItMg5HdVeN7CGpKJiBEyS4ofpSronRAerrJhucAJScpF646wi0NhRPuCtu6QnrxFChZEr6h62rxaanQ0XuIVDM8+oo4wpDIoBczqC7Ab3alzia3VkuOn8A9tOPWuNfksPwQGcLbbQGFhStYqRxcgMbbggKSBnzKRUV9JsHmFlrKCZujsugj+AHp+aboFS+zBV4NfJBUM2pU9KvzvfI8419h2t+hSSVPWauOQpFw7tii35PD7shTRo3IRIwjWanM9ZJDlFAtKaAbSqnvTfcZzeDIHoxndOl8mZLc8aFAiD+OP4G+TxWHhh0jwBn/fcCaIWKufgLmFlheqfJKURFv7EaxlC74GJ8OpZn0HxudAq9BpdcH1U6mDLHcOmc89dutkGYNevv57nnld4FB1xs/pFqzm2GQ/AFn0U0pcwUtG00XrrIwLBv7Fi0hAr6oBV0G5ltmoix1STWMVwz5Ct1DRQecvjsamjSY6kueESUobJHjNvd6f4uQb9pOFbhLQYajFLOlzexGyuNcOt36AsSQAeduslmsSvsP1E2fLUAH7+eX1XIt+I5JI4oW1deX+WcMjEnHSToX9pgJj2QnSsa7fZdTTlE+an52L4VC4kduids+1ETfNjfnTTw6d/TPuW1y2L002/UfZPU5EaKer4A8lljQiOWZxIeXGk1HT8t0XCByZ8fISousxd9SqZgP5nuEjBHoSttz2LvrJ/AUEkZRiSr/dCBtchNtMD2IpXFlmedc1ReBSuTyEUeWTFOEV/+xavaYGiCg8pPG4ZcCHTPcZa1cXx5Ia+lGfrpL8BM6qvTGIs7VHUVX0OcYliXPKR4+jhbqQ+6FMynSHi2IMRdhyTGFnTyz/FLpOsxOqZRNvi8E0DJ84oT6forPikuViayQxBo/hc5r5cZyJ+0RjYMCIn2hO2zltwdhLs2r9wj+A04yCPdlDgAA"
probe = unpack(PROBE_B64)
p = predict_proba(final_model, probe.text.tolist()).argmax(1)
probe["pred"] = [id2label[i] for i in p]; probe["ok"] = probe.pred == probe.category_id
by_script = probe.groupby("script").ok.mean()
tests = {"franco_probe": by_script.round(4).to_dict()}
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(by_script.index, by_script.values * 100, color="#0f766e")
for i, v in enumerate(by_script.values):
    ax.text(i, v * 100 + 1, f"{v:.0%}", ha="center")
ax.set_ylim(0, 110); ax.set_ylabel("% correct"); ax.set_title("New sentences the model never saw")
plt.tight_layout(); plt.show()
print("Wrong ones:")
display(probe.loc[~probe.ok, ["text", "category_id", "pred"]])

if scenario is not None:
    sp = predict_proba(final_model, scenario.description_ar.astype(str).tolist()).argmax(1)
    acc = accuracy_score(scenario.category_id, [id2label[i] for i in sp])
    tests["scenario_accuracy"] = round(acc, 4); print(f"scenario test accuracy: {acc:.1%} on {len(scenario)} complaints")
    sc = scenario.assign(pred=[id2label[i] for i in sp])
    display(sc.assign(ok=sc.pred == sc.category_id).groupby("category_id").ok.mean().round(3).to_frame("accuracy"))

### RESULTS 2: the final model on new complaints

In [ ]:
# ===== RESULTS 2: the final model on new complaints (Franco / English / scenario) =====
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, HTML
import os; os.makedirs("results", exist_ok=True)
from sklearn.metrics import accuracy_score

_p = predict_proba(final_model, probe.text.tolist()).argmax(1)
_probe = probe.assign(pred=[id2label[i] for i in _p])
_probe["correct"] = _probe.pred == _probe.category_id
_by = _probe.groupby("script").correct.mean()

_cards = "".join(
    f'<div style="background:#0f766e;color:white;border-radius:12px;padding:14px 22px;min-width:160px">'
    f'<div style="font-size:14px;opacity:.85">{k.title()} sentences</div><div style="font-size:36px;font-weight:700">{v:.0%}</div></div>'
    for k, v in _by.items())
if "scenario" in globals() and scenario is not None:
    _sp = predict_proba(final_model, scenario.description_ar.astype(str).tolist()).argmax(1)
    _sc = scenario.assign(pred=[id2label[i] for i in _sp])
    _sc_acc = accuracy_score(_sc.category_id, _sc.pred)
    _cards += (f'<div style="background:#115e59;color:white;border-radius:12px;padding:14px 22px;min-width:160px">'
               f'<div style="font-size:14px;opacity:.85">Scenario test ({len(_sc)})</div><div style="font-size:36px;font-weight:700">{_sc_acc:.1%}</div></div>')
display(HTML(f'<div style="font-family:sans-serif;display:flex;gap:16px;flex-wrap:wrap;margin:8px 0">{_cards}</div>'
             '<p style="font-family:sans-serif;color:#475569">% of complaints classified correctly. None of these sentences were used in training.</p>'))

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.bar(_by.index, _by.values * 100, color="#0f766e")
for i, v in enumerate(_by.values):
    ax.text(i, v * 100 + 1, f"{v:.0%}", ha="center")
ax.set_ylim(0, 110); ax.set_ylabel("% correct"); ax.set_title("New Franco / English sentences")
plt.tight_layout(); fig.savefig("results/new_sentences.png", dpi=150); plt.show()

print("Sentences the model got wrong:")
display(_probe.loc[~_probe.correct, ["text", "category_id", "pred"]].rename(columns={"category_id": "true"}))
if "scenario" in globals() and scenario is not None:
    print("Scenario test, accuracy per category:")
    display(_sc.assign(correct=_sc.pred == _sc.category_id).groupby("category_id").correct.mean().round(3).to_frame("accuracy"))

## 10. Try your own sentences

In [ ]:
def classify(text, k=3):
    proba = predict_proba(final_model, [text])[0]
    return [(id2label[i], round(float(proba[i]), 3)) for i in proba.argsort()[::-1][:k]]

examples = ["المية مقطوعة من امبارح", "el kahraba ati3a fel 3emara", "fe ri7et gaz gamda",
            "the traffic light is stuck on red", "الزبالة متكومة جنب المدرسة", "el app by2fel lewa7do"]
display(pd.DataFrame([{"complaint": t, "category": classify(t)[0][0], "confidence": classify(t)[0][1],
                       "2nd": classify(t)[1][0]} for t in examples]))

summary = {"training sentences": len(df), "categories": len(LABELS), "best epochs": BEST_EPOCHS,
           "validation macro-F1": validation["macro_f1"], "validation accuracy": validation["accuracy"]}
summary.update({f"new sentences ({k})": f"{v:.0%}" for k, v in tests["franco_probe"].items()})
if "scenario_accuracy" in tests:
    summary["scenario test accuracy"] = f"{tests['scenario_accuracy']:.1%}"
display(pd.DataFrame(summary.items(), columns=["result", "value"]))

### RESULTS 3: examples, summary table, and oasis_results.zip (charts + numbers)

In [ ]:
# ===== RESULTS 3: try complaints + one summary table (also saved as oasis_results.zip) =====
import os, json, shutil, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, HTML

def top3(text):
    pr = predict_proba(final_model, [text])[0]
    idx = pr.argsort()[::-1][:3]
    return [(id2label[i], float(pr[i])) for i in idx]

_examples = ["المية مقطوعة من امبارح", "el kahraba ati3a fel 3emara", "fe ri7et gaz gamda fel sellem",
             "the traffic light is stuck on red", "الزبالة متكومة جنب المدرسة", "el app by2fel lewa7do",
             "عمود النور مطفي والشارع ضلمة", "el bala3a taf7a odam el 3emara"]
_rows = []
for t in _examples:
    r = top3(t)
    _rows.append({"complaint": t, "category": r[0][0], "confidence": f"{r[0][1]:.0%}", "2nd choice": f"{r[1][0]} ({r[1][1]:.0%})"})
display(pd.DataFrame(_rows))

_summary = {"Model": MODEL_NAME, "Training sentences": len(df), "Categories": len(LABELS)}
if "RESULTS" in globals():
    _summary.update({"Accuracy (unseen 15%)": f"{RESULTS['accuracy']:.1%}", "Macro-F1 (unseen 15%)": f"{RESULTS['macro_f1']:.3f}",
                     "Best epoch": RESULTS["best_epoch"]})
if "_by" in globals():
    _summary.update({f"New {k.title()} sentences": f"{v:.0%}" for k, v in _by.items()})
if "_sc_acc" in globals():
    _summary["Scenario test accuracy"] = f"{_sc_acc:.1%}"
_table = pd.DataFrame(_summary.items(), columns=["Result", "Value"])
display(HTML("<h3 style='font-family:sans-serif'>Model results</h3>" + _table.to_html(index=False)))

# Save the numbers (the charts were saved by the cells above), then download everything.
os.makedirs("results", exist_ok=True)
_table.to_csv("results/summary.csv", index=False)
pd.DataFrame(_rows).to_csv("results/examples.csv", index=False)
shutil.make_archive("oasis_results", "zip", "results")
try:
    from google.colab import files
    files.download("oasis_results.zip")
except Exception as e:
    print("download from the Files panel: oasis_results.zip", e)

## 11. Save and download `oasis_bert_model.zip` (about 300 MB)
Unzip it inside the platform folder `ml/artifacts/`, so that you get `ml/artifacts/bert/config.json`. Then run `run.bat` again: it installs the BERT libraries once and the website switches to BERT by itself.

In [ ]:
OUT = "bert"
if os.path.exists(OUT): shutil.rmtree(OUT)
final_model.half()   # 16-bit weights: half the download size, same predictions in practice
final_model.save_pretrained(OUT, safe_serialization=True)
tokenizer.save_pretrained(OUT)
shutil.copy("oasis_franco.py", f"{OUT}/oasis_franco.py")
info = {
    "model_version": "bert-1.0.0",
    "base_model": MODEL_NAME,
    "trained_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "max_len": MAX_LEN, "epochs": BEST_EPOCHS, "labels": LABELS,
    "input_rule": "text + ' ' + gloss(text)  (oasis_franco.py)",
    "training_texts": len(df), "validation": validation, "tests": tests,
    "transformers_version": __import__("transformers").__version__, "torch_version": torch.__version__,
}
json.dump(info, open(f"{OUT}/model_info.json", "w"), ensure_ascii=False, indent=2)
shutil.make_archive("oasis_bert_model", "zip", ".", OUT)
print(f"oasis_bert_model.zip: {os.path.getsize('oasis_bert_model.zip') / 1e6:.0f} MB")
files.download("oasis_bert_model.zip")

## 12. RESULTS DISPLAY: the saved model, shown clearly
These 4 cells load the `oasis_bert_model.zip` that step 11 just saved (no training) and show the accuracy cards, the charts and the mistakes.
They also work on their own later: open a new Colab, drag `oasis_bert_model.zip` (and the data zip, optional) into the Files panel, and run only steps 12.1 to 12.4.

### 12.1 Load the saved model

In [ ]:
# CELL 1: load the saved model (no training). Put oasis_bert_model.zip (and the data zip, optional) in the Files panel first.
import os, io, re, sys, json, glob, gzip, base64, zipfile
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from IPython.display import display, HTML

MODEL_ZIP = (glob.glob("/content/**/oasis_bert_model*.zip", recursive=True) or [None])[0]
assert MODEL_ZIP, "Drag oasis_bert_model.zip into the Files panel (folder icon on the left) and wait for the upload to finish."
with zipfile.ZipFile(MODEL_ZIP) as z:
    z.extractall("/content/model")
MODEL_DIR = os.path.dirname(glob.glob("/content/model/**/config.json", recursive=True)[0])
sys.path.insert(0, MODEL_DIR)
from oasis_franco import gloss

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
INFO = json.load(open(f"{MODEL_DIR}/model_info.json"))
tok = AutoTokenizer.from_pretrained(MODEL_DIR)
bert = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).float().to(DEVICE).eval()
LABELS = [bert.config.id2label[i] for i in range(bert.config.num_labels)]

@torch.no_grad()
def predict(texts, batch=64):
    out = []
    for s in range(0, len(texts), batch):
        enc = tok([f"{t} {gloss(t)}".strip() for t in texts[s:s + batch]], truncation=True,
                  max_length=INFO.get("max_len", 96), padding=True, return_tensors="pt").to(DEVICE)
        out.append(torch.softmax(bert(**enc).logits, -1).cpu().numpy())
    return np.vstack(out)

print("model loaded:", INFO["base_model"], "|", len(LABELS), "categories | device:", DEVICE)

### 12.2 Results in numbers

In [ ]:
# CELL 2: the results in numbers (tests on complaints the model never saw)
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
os.makedirs("/content/results", exist_ok=True)
def card(title, value, color="#0f766e"):
    return (f'<div style="background:{color};color:white;border-radius:12px;padding:14px 22px;min-width:170px">'
            f'<div style="font-size:14px;opacity:.85">{title}</div><div style="font-size:38px;font-weight:700">{value}</div></div>')

# 1) Tests on sentences the model never saw
PROBE = pd.read_csv(io.BytesIO(gzip.decompress(base64.b64decode("H4sIAAAAAAAC/31XwZKrOAy8v6/w7V1ygpql9mumBBbgwtis7QxLvn5bdmAgL7OXSQbcttVqtZTE/6ZbR4kHH7ZPo2+xC2ZJv9gqClp5TbPC95ZaNYwUKnKkZtpIOdbMPf5xKlKdmN1txTbh0zJNtz6Q6/yvntUHpQJIFLhSPTYb+GECybarj4ne4PAqY1q8iKOcsVK0lNENv4WkkQP/jorUctfaskojTnb8xUHpYDiqlju6R5YVZmFlohK8ccN5N3aDNXH8NeNYS9WGuPZ4I+JvFa34Ws/Cz2SfSH9PNPDr5VeqwE9NTmcGre8mNXnrJZxaSKP0E74BzT1vQnWbHD3Y4joNDQQ6Lce3sEQLog+sOj8vlhPbDYFvyjjl70HFFJhfDtyDPdHNw4YzTFIBF+Aaid9PWwLHeA/8af16umqkaSRwBHpUu+FvTRZBR3m476upFq38z0aahjEd61uqQB83XHLfUp0pTNAANwdr7/bZn0l2k/cKr/CpemMBHyEKctM7+ImJlixSAy4q2iSLcrKkHmTk7AXT0C3yymE/NN85Uuixva1zSTzLZ6C5LVHFgfUVhXJKVF660UOwM03R+3BdJZfOD5Q1LofVUieiVfcFqUVseUVJbkGeYploDNSibnL11CUlEgwSm6h2dFv8+k6APQolih76KGWaBcxQRWZAk53fI2cRDZK2nytMzC24a/LBwuvfb4ErI1VRFMBdCqYzKQtXYlszOSJqf3f6ij6FmmhCsfaF0DDRY5MCxIM+P2kR6/hF3SlGIATQ8lZBb3WRnqgXiUjUvi5vKIDHhxTF4Ff6PqjQs5+fCwEUI732soXE0nlvJXdIo+/7PcLI4ct0rIB4Qk5xPVgEKaloK/EDu4kFQ9tWTvWx9TUEHRN/elhdf6qFhlbeOFfVsclfPTU5xkDR9D8ABZDXzynQLDaEreA7RT3wM++rH6CtccWE9hcSrThghGOCJA7BtJZf0ad4n6WGzrFSoArkgliK8gDiK+7fwaqd1EOMpzvnkNQsBOH7bM/WAUMSA3oDbEjL6i2bzoaW0bKj0qfKVd6BnG+93pCwqOLKS8ppDB5bIaUr8xQvqItMg5HdVeN7CGpKJiBEyS4ofpSronRAerrJhucAJScpF646wi0NhRPuCtu6QnrxFChZEr6h62rxaanQ0XuIVDM8+oo4wpDIoBczqC7Ab3alzia3VkuOn8A9tOPWuNfksPwQGcLbbQGFhStYqRxcgMbbggKSBnzKRUV9JsHmFlrKCZujsugj+AHp+aboFS+zBV4NfJBUM2pU9KvzvfI8419h2t+hSSVPWauOQpFw7tii35PD7shTRo3IRIwjWanM9ZJDlFAtKaAbSqnvTfcZzeDIHoxndOl8mZLc8aFAiD+OP4G+TxWHhh0jwBn/fcCaIWKufgLmFlheqfJKURFv7EaxlC74GJ8OpZn0HxudAq9BpdcH1U6mDLHcOmc89dutkGYNevv57nnld4FB1xs/pFqzm2GQ/AFn0U0pcwUtG00XrrIwLBv7Fi0hAr6oBV0G5ltmoix1STWMVwz5Ct1DRQecvjsamjSY6kueESUobJHjNvd6f4uQb9pOFbhLQYajFLOlzexGyuNcOt36AsSQAeduslmsSvsP1E2fLUAH7+eX1XIt+I5JI4oW1deX+WcMjEnHSToX9pgJj2QnSsa7fZdTTlE+an52L4VC4kduids+1ETfNjfnTTw6d/TPuW1y2L002/UfZPU5EaKer4A8lljQiOWZxIeXGk1HT8t0XCByZ8fISousxd9SqZgP5nuEjBHoSttz2LvrJ/AUEkZRiSr/dCBtchNtMD2IpXFlmedc1ReBSuTyEUeWTFOEV/+xavaYGiCg8pPG4ZcCHTPcZa1cXx5Ia+lGfrpL8BM6qvTGIs7VHUVX0OcYliXPKR4+jhbqQ+6FMynSHi2IMRdhyTGFnTyz/FLpOsxOqZRNvi8E0DJ84oT6forPikuViayQxBo/hc5r5cZyJ+0RjYMCIn2hO2zltwdhLs2r9wj+A04yCPdlDgAA"))))
PROBE["pred"] = [LABELS[i] for i in predict(PROBE.text.tolist()).argmax(1)]
PROBE["correct"] = PROBE.pred == PROBE.category_id
by_script = PROBE.groupby("script").correct.mean()

SCEN = None
for f in glob.glob("/content/**/*.zip", recursive=True):
    if "oasis_bert_model" in f: continue
    with zipfile.ZipFile(f) as z:
        n = z.namelist()
        a = [x for x in n if x.endswith("scenario_complaints_with_answer_keys_SYNTHETIC.csv")]
        b = [x for x in n if x.endswith("02_SUPPORTING_CSV/scenario_complaints.csv")]
        c = [x for x in n if x.endswith("test_complaint_category_labels.csv")]
        if a:
            SCEN = pd.read_csv(io.BytesIO(z.read(a[0]))).rename(columns={"answer_key_category_id": "category_id"})
        elif b and c:
            SCEN = pd.read_csv(io.BytesIO(z.read(b[0])), encoding="utf-8-sig").merge(
                pd.read_csv(io.BytesIO(z.read(c[0])), encoding="utf-8-sig"), on="request_id")
    if SCEN is not None: break

v = INFO["validation"]
cards = card("Accuracy (unseen 15%)", f"{v['accuracy']:.1%}") + card("Macro-F1 (unseen 15%)", f"{v['macro_f1']:.3f}", "#115e59")
cards += "".join(card(f"New {k.title()} sentences", f"{x:.0%}", "#334155") for k, x in by_script.items())
if SCEN is not None:
    sp = predict(SCEN.description_ar.astype(str).tolist()).argmax(1)
    SCEN["pred"] = [LABELS[i] for i in sp]
    cards += card(f"Scenario test ({len(SCEN)})", f"{accuracy_score(SCEN.category_id, SCEN.pred):.1%}", "#334155")
display(HTML(f'<h2 style="font-family:sans-serif">Oasis BERT complaint classifier: results</h2>'
             f'<div style="font-family:sans-serif;display:flex;gap:14px;flex-wrap:wrap">{cards}</div>'
             f'<p style="font-family:sans-serif;color:#475569">{INFO["base_model"]} fine-tuned on {INFO["training_texts"]:,} sentences, '
             f'{len(LABELS)} categories. Every number above is on complaints the model never saw in training.</p>'))

### 12.3 Charts

In [ ]:
# CELL 3: charts (learning curve, F1 per category, confusion matrix)
# 2) Learning curve (round 1)
h = pd.DataFrame(v["history"])
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(h.epoch, h.loss, marker="o", color="#64748b"); ax[0].set_title("Training loss (lower is better)"); ax[0].set_xlabel("epoch")
ax[1].plot(h.epoch, h.val_acc, marker="s", color="#d97706", label="accuracy")
ax[1].plot(h.epoch, h.val_macro_f1, marker="o", color="#0f766e", label="macro-F1")
ax[1].axvline(v["best_epochs"], ls="--", color="grey"); ax[1].set_ylim(0, 1.02); ax[1].legend()
ax[1].set_title(f"Score on unseen sentences per epoch (best = {v['best_epochs']})"); ax[1].set_xlabel("epoch")
plt.tight_layout(); fig.savefig("/content/results/learning_curve.png", dpi=150); plt.show()

# 3) Per-category F1 and confusion matrix on the scenario complaints (never trained on)
if SCEN is not None:
    labs = sorted(set(SCEN.category_id) | set(SCEN.pred))
    rep = pd.DataFrame(classification_report(SCEN.category_id, SCEN.pred, labels=labs, zero_division=0, output_dict=True)).T
    pc = rep.loc[labs].sort_values("f1-score")
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(pc.index, pc["f1-score"], color=["#dc2626" if x < 0.7 else "#0f766e" for x in pc["f1-score"]])
    for i, x in enumerate(pc["f1-score"]): ax.text(x + 0.01, i, f"{x:.2f}", va="center")
    ax.set_xlim(0, 1.1); ax.set_title("F1 per category, scenario test (red = below 0.70)")
    plt.tight_layout(); fig.savefig("/content/results/f1_per_category.png", dpi=150); plt.show()
    cm = confusion_matrix(SCEN.category_id, SCEN.pred, labels=labs)
    fig, ax = plt.subplots(figsize=(10, 9)); ax.imshow(cm, cmap="Greens")
    ax.set_xticks(range(len(labs))); ax.set_xticklabels(labs, rotation=60, ha="right"); ax.set_yticks(range(len(labs))); ax.set_yticklabels(labs)
    for i in range(len(labs)):
        for j in range(len(labs)):
            if cm[i, j]: ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xlabel("predicted"); ax.set_ylabel("true category"); ax.set_title("Confusion matrix, scenario test (diagonal = correct)")
    plt.tight_layout(); fig.savefig("/content/results/confusion_matrix.png", dpi=150); plt.show()
    display(rep.round(3).rename(columns={"support": "complaints"}))
else:
    print("(Put the data zip in the Files panel too, to also see the 420-complaint scenario test and its confusion matrix.)")

### 12.4 Examples and mistakes

In [ ]:
# CELL 4: examples and the sentences the model got wrong
# 4) Examples and mistakes
ex = ["المية مقطوعة من امبارح", "el kahraba ati3a fel 3emara", "fe ri7et gaz gamda fel sellem", "the traffic light is stuck on red",
      "الزبالة متكومة جنب المدرسة", "el app by2fel lewa7do", "عمود النور مطفي والشارع ضلمة", "el bala3a taf7a odam el 3emara"]
P = predict(ex)
display(pd.DataFrame({"complaint": ex, "category": [LABELS[i] for i in P.argmax(1)], "confidence": [f"{p.max():.0%}" for p in P]}))
print("New Franco/English sentences the model got wrong:")
display(PROBE.loc[~PROBE.correct, ["text", "category_id", "pred"]].rename(columns={"category_id": "true"}))